# Beta-Distribution GDD Model — *Prunus yedoensis* (Somei-yoshino)

Trains `GlobalBetaGDDModel` on the GMU Japan yedoensis cherry blossom dataset
and explores the learned behaviour.

`GlobalBetaGDDModel` is a fully differentiable two-stage phenology model:

1. **Chilling** — per-day contribution = beta PDF evaluated at the normalised
   temperature, rescaled to peak at 1.  Both shape parameters `α > 1` and
   `β > 1` are enforced by construction, guaranteeing a **unimodal** (bell-shaped)
   chilling response.  The mode corresponds to the learned optimal chilling
   temperature `T_opt`.
2. **Forcing** — cumulative `relu(T − t_base)` after the chilling gate is open,
   identical to all other GDD-based models in this codebase.

All parameters (`α`, `β`, `t_base`, chilling threshold, forcing threshold) are
optimised by gradient descent against binary cross-entropy on soft step-function
labels.

## Config

In [ ]:
import torch

OBS_KEY = 'gmu_0'
CUTOFF  = 2015          # train: 1986–2014, test: 2015–present
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

# Beta-GDD temperature range for normalisation
T_LOW  = -5.0   # °C mapped to 0
T_HIGH = 20.0   # °C mapped to 1

# Style helpers
_MONTH_STARTS = [0, 31, 61, 92, 122, 153, 181, 212, 243, 273, 304, 334]
_MONTH_LABELS = ['Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar',
                 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep']
MODEL_COLOR = '#1a6b3c'

def _month_ticks(n):
    t = [d for d in _MONTH_STARTS if d < n]
    l = [_MONTH_LABELS[i] for i, d in enumerate(_MONTH_STARTS) if d < n]
    return t, l

def _style_ax(ax, n, ylabel='', xlabel=False):
    ax.set_xlim(0, n - 1)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    t, l = _month_ticks(n)
    ax.set_xticks(t)
    ax.set_xticklabels(l if xlabel else [], fontsize=8)
    if xlabel:
        ax.set_xlabel('Month (season from Oct 1)', fontsize=9)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F

from pysephone.constants import KEY_FEATURES
from pysephone.dataset.dataset import Dataset
from pysephone.dataset.util.calendar import Calendar
from pysephone.dataset.util.feature_cache import FeatureCache
from pysephone.evaluation.regression import SingleTargetRegression
from pysephone.models.beta_gdd import GlobalBetaGDDModel

## 1. Load data

Season starts 1 October, runs 365 days — capturing the full chill–force cycle
for *Prunus yedoensis* (Somei-yoshino cherry).  Only yedoensis (species_id 0)
is selected from the combined YS dataset.

In [ ]:
DATA_KEYS = ['temperature_2m_mean']

cal    = Calendar(default_start='10-01', default_length=365)
_cache = FeatureCache.load(FeatureCache.default_path('GMU_Cherry_Japan_YS', DATA_KEYS))
ds_ys  = Dataset.load('GMU_Cherry_Japan_YS', calendar=cal, feature_providers=[_cache])
ds_y   = ds_ys.select_species([('GMU_cherry', 0)])   # Somei-yoshino only

target_fn = lambda s: s['observations'][OBS_KEY]

print(f'Dataset: {len(ds_y)} samples | '
      f'years {min(ds_y.years)}–{max(ds_y.years)} | '
      f'locations {len(ds_y.locations)}')

## 2. Temporal train / test split

Train: 1986–2014, test: 2015–present.

In [ ]:
years_trn = [y for y in ds_y.years if y < CUTOFF]
years_tst = [y for y in ds_y.years if y >= CUTOFF]

ds_trn = ds_y.select_years(years_trn)
ds_tst = ds_y.select_years(years_tst)

print(f'Train: {len(ds_trn)} samples  ({min(years_trn)}–{max(years_trn)})')
print(f'Test:  {len(ds_tst)} samples  ({min(years_tst)}–{max(years_tst)})')

## 3. Fit `GlobalBetaGDDModel`

Five learnable scalar parameters:

| Parameter | Role |
|---|---|
| `α`, `β` | Shape of the beta chilling response (both constrained > 1) |
| `t_base` | GDD base temperature |
| `tt_unit_threshold` | Cumulative chilling units required before forcing begins |
| `gdd_threshold` | Cumulative GDD required to trigger bloom |

In [ ]:
MODEL_KWARGS = dict(
    t_low            = T_LOW,
    t_high           = T_HIGH,
    alpha_init       = 2.0,
    beta_init        = 2.0,
    learn_alpha_beta = True,
    learn_t_base     = True,
    t_base_init      = 5.0,
    learn_thresholds = True,
)

model, fit_info = GlobalBetaGDDModel.fit(
    target_fn        = target_fn,
    dataset          = ds_trn,
    model_kwargs     = MODEL_KWARGS,
    num_epochs       = 1000,
    batch_size       = 512,
    val_period       = 10,
    optimizer        = 'adam',
    optimizer_kwargs = dict(lr=1e-3, weight_decay=1e-4),
    scheduler_step_size      = 100,
    scheduler_decay          = 0.9,
    early_stopping           = True,
    early_stopping_patience  = 10,
    early_stopping_min_delta = 1e-3,
    device  = DEVICE,
    verbose = True,
)
print('Training complete.')

In [ ]:
# Training / validation loss curves
epochs   = [e['epoch'] for e in fit_info['epochs']]
trn_loss = [e['loss']  for e in fit_info['epochs']]
val_ep   = [e['epoch'] for e in fit_info['epochs'] if 'val' in e]
val_loss = [e['val']['loss'] for e in fit_info['epochs'] if 'val' in e]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(epochs, trn_loss, label='Train', color='steelblue', lw=1.5)
if val_loss:
    ax.plot(val_ep, val_loss, label='Val', color='coral', lw=1.5)
    ax.axhline(min(val_loss), color='grey', ls='--', lw=0.8)
ax.set_xlabel('Epoch', fontsize=9)
ax.set_ylabel('BCE Loss', fontsize=9)
ax.set_title('GlobalBetaGDDModel — training loss', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Learned parameters
alpha   = model.alpha.item()
beta    = model.beta.item()
t_base  = model._gdd._tb.item()
tt_thr  = model._tt_unit_threshold.threshold.item()
gdd_thr = model._gdd_threshold.threshold.item()

# Optimal chilling temperature: mode of beta(α, β) mapped back to °C
mode_norm = (alpha - 1) / (alpha + beta - 2)
T_opt     = T_LOW + (T_HIGH - T_LOW) * mode_norm

print(f'α (alpha):               {alpha:.3f}')
print(f'β (beta):                {beta:.3f}')
print(f'Optimal chill temp T_opt:{T_opt:.1f} °C  (mode of beta at T_norm = {mode_norm:.3f})')
print(f't_base (GDD):            {t_base:.2f} °C')
print(f'Chilling threshold:      {tt_thr:.3f}  ≈ {tt_thr * 200:.1f} cum. units')
print(f'Forcing threshold:       {gdd_thr:.3f}  ≈ {gdd_thr * 500:.1f} GDD')

## 4. Evaluate

In [ ]:
result = SingleTargetRegression.run(
    model         = model,
    dataset_train = ds_trn,
    dataset_test  = ds_tst,
    target_fn     = target_fn,
    run_name      = 'beta_gdd_yedoensis',
)

m = result.compute_metrics()
rows = []
for split in ('train', 'test'):
    rows.append({
        'Split': split,
        'N':     m[split]['n'],
        'RMSE':  round(m[split]['rmse'], 2),
        'MAE':   round(m[split]['mae'],  2),
        'Bias':  round(m[split]['bias'], 2),
        'R²':    round(m[split]['r2'],   3),
    })
pd.DataFrame(rows).set_index('Split')

In [ ]:
fig = result.plot_scatter(title='GlobalBetaGDDModel — Somei-yoshino — predicted vs observed')
plt.tight_layout()
plt.show()

fig = result.plot_residuals_over_time(title='GlobalBetaGDDModel — Somei-yoshino — residuals over time')
plt.tight_layout()
plt.show()

## 5. Learned beta chilling response

The chilling contribution at temperature `T` is the unnormalised beta PDF
evaluated at `T_norm = (T − t_low) / (t_high − t_low)`, divided by its own
maximum so it peaks at 1.0.  The mode (`T_opt`) is the optimal chilling temperature.

In [ ]:
@torch.no_grad()
def plot_beta_response(model, t_range=(-10, 30), n=500):
    """Plot the learned beta-distribution chilling response vs temperature."""
    model.eval()
    eps   = 1e-6
    t_low  = model.t_low_eff.item()
    t_high = model.t_high_eff.item()
    alpha  = model.alpha.item()
    beta   = model.beta.item()

    t_grid  = np.linspace(t_range[0], t_range[1], n)
    T_norm  = np.clip((t_grid - t_low) / (t_high - t_low + eps), eps, 1 - eps)

    log_contrib = (alpha - 1) * np.log(T_norm) + (beta - 1) * np.log(1 - T_norm)
    mode_norm   = (alpha - 1) / (alpha + beta - 2)
    log_max     = (alpha - 1) * np.log(mode_norm) + (beta - 1) * np.log(1 - mode_norm)
    contrib     = np.exp(log_contrib - log_max)

    T_opt = t_low + (t_high - t_low) * mode_norm

    fig, ax = plt.subplots(figsize=(8, 4))
    fig.suptitle(
        f'Learned beta chilling response  (α={alpha:.2f}, β={beta:.2f})',
        fontsize=11, fontweight='bold',
    )

    ax.fill_between(t_grid, contrib, color=MODEL_COLOR, alpha=0.15)
    ax.plot(t_grid, contrib, color=MODEL_COLOR, lw=2.5,
            label=f'beta PDF  (α={alpha:.2f}, β={beta:.2f})')
    ax.axvline(T_opt, color='grey', lw=1.2, ls='--',
               label=f'T_opt = {T_opt:.1f} °C')
    ax.axvline(t_low,  color='lightgrey', lw=0.8, ls=':',
               label=f't_low = {t_low:.1f} °C')
    ax.axvline(t_high, color='lightgrey', lw=0.8, ls=':',
               label=f't_high = {t_high:.1f} °C')
    ax.set_xlabel('Temperature (°C)', fontsize=10)
    ax.set_ylabel('Chilling contribution', fontsize=10)
    ax.set_ylim(-0.05, 1.1)
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

    print(f'α = {alpha:.3f},  β = {beta:.3f}')
    print(f't_low = {t_low:.1f} °C,  t_high = {t_high:.1f} °C')
    print(f'Optimal chilling temperature T_opt = {T_opt:.1f} °C')
    print(f'Contribution is > 0.5 over '
          f'[{t_grid[contrib > 0.5].min():.1f}, {t_grid[contrib > 0.5].max():.1f}] °C')


plot_beta_response(model)

## 6. Season dynamics on held-out samples

Five-panel plot showing each stage of the model's computation for representative
test samples (earliest, median, and latest bloom year).

In [ ]:
def pick_sample(ds, year):
    for item in ds.iter_items():
        if item['year'] == year:
            return item
    raise ValueError(f'No sample for year {year}')


# Pick earliest / median / latest bloom in test set
test_obs = []
for year in sorted(ds_tst.years):
    s = pick_sample(ds_tst, year)
    obs_date = s['observations'][OBS_KEY]
    ix = int((np.datetime64(obs_date, 'D')
              - np.datetime64(s['season_start'], 'D'))
             / np.timedelta64(1, 'D'))
    test_obs.append((year, ix))
test_obs.sort(key=lambda x: x[1])
n_tst = len(test_obs)
years_to_show  = [test_obs[0][0], test_obs[n_tst // 2][0], test_obs[-1][0]]
labels_to_show = ['Earliest bloom', 'Median bloom', 'Latest bloom']
print('Years:', years_to_show)

In [ ]:
@torch.no_grad()
def plot_season_dynamics(model, sample, obs_ix=None, title=''):
    model.eval()
    device = next(model.parameters()).device
    t_s    = GlobalBetaGDDModel.cast_to_tensor(sample, device=device)
    batch  = GlobalBetaGDDModel.collate_fn([t_s])
    ixs, info = model(batch)

    pred_ix    = int(ixs[0].item() + 0.5)
    ts         = sample[KEY_FEATURES]['temperature_2m_mean'].astype(float)
    n          = len(ts)
    days       = np.arange(n)
    tt_contrib = info['tt_contrib'][0].cpu().numpy()
    tt_units   = info['tt_units'][0].cpu().numpy()
    tt_reached = info['tt_reached'][0].cpu().numpy()
    gdd_cum    = info['gdd_cum'][0].cpu().numpy()
    ps         = info['ps'][0].cpu().numpy()

    tt_thr_raw = model._tt_unit_threshold.threshold.item()
    gdd_thr_raw = model._gdd_threshold.threshold.item()
    t_base_val  = model._gdd._tb.item()
    # Thresholds in natural units (SoftThreshold bounds: 0–200 and 0–500)
    tt_thr_nat  = tt_thr_raw * 200
    gdd_thr_nat = gdd_thr_raw * 500

    WARM = '#f4a46240'; COLD = '#7eb8d440'; TLINE = '#c0392b'

    fig, axes = plt.subplots(5, 1, figsize=(11, 12), sharex=True)
    if title:
        fig.suptitle(title, fontsize=11, fontweight='bold')

    # 1. Temperature
    ax = axes[0]
    ax.axhline(0, color='#cccccc', lw=0.8)
    ax.axhline(t_base_val, color='navy', lw=1.0, ls=':',
               label=f't_base = {t_base_val:.1f} °C')
    ax.fill_between(days, ts, 0, where=(ts >= 0), color=WARM)
    ax.fill_between(days, ts, 0, where=(ts  < 0), color=COLD)
    ax.plot(days, ts, color=TLINE, lw=0.9)
    if obs_ix is not None:
        ax.axvline(obs_ix,  color='black',       lw=1.2, ls='-',  alpha=0.7, label='Observed')
    ax.axvline(pred_ix, color=MODEL_COLOR, lw=1.2, ls='--', alpha=0.9, label='Predicted')
    _style_ax(ax, n, ylabel='T (°C)')
    ax.legend(fontsize=7.5, loc='upper right')

    # 2. Per-day chilling contribution (beta response)
    ax = axes[1]
    ax.fill_between(days, tt_contrib, color=MODEL_COLOR, alpha=0.3)
    ax.plot(days, tt_contrib, color=MODEL_COLOR, lw=1.5)
    if obs_ix is not None:
        ax.axvline(obs_ix,  color='black',       lw=1.0, ls='-',  alpha=0.5)
    ax.axvline(pred_ix, color=MODEL_COLOR, lw=1.0, ls='--', alpha=0.7)
    _style_ax(ax, n, ylabel='Chill contrib')
    ax.set_ylim(-0.02, 1.08)
    ax.set_title('Per-day chilling contribution (beta PDF response)', fontsize=8)

    # 3. Cumulative chilling + soft threshold
    ax = axes[2]
    ax.plot(days, tt_units, color=MODEL_COLOR, lw=1.8, label='Cumulative chilling')
    ax.axhline(tt_thr_nat, color=MODEL_COLOR, lw=1.0, ls=':',
               label=f'Threshold ≈ {tt_thr_nat:.1f}')
    ax2 = ax.twinx()
    ax2.plot(days, tt_reached, color=MODEL_COLOR, lw=1.0, ls='--', alpha=0.5)
    ax2.set_ylabel('tt_reached', fontsize=8, color=MODEL_COLOR)
    ax2.set_ylim(-0.05, 1.1)
    ax2.tick_params(labelsize=7, colors=MODEL_COLOR)
    _style_ax(ax, n, ylabel='Cum. chill units')
    ax.legend(fontsize=7.5)

    # 4. Cumulative masked GDD
    ax = axes[3]
    COLOR_F = '#c0392b'
    ax.fill_between(days, gdd_cum, color=COLOR_F, alpha=0.25)
    ax.plot(days, gdd_cum, color=COLOR_F, lw=1.8, label='Cumulative masked GDD')
    ax.axhline(gdd_thr_nat, color=COLOR_F, lw=1.0, ls=':',
               label=f'Threshold ≈ {gdd_thr_nat:.1f}')
    _style_ax(ax, n, ylabel='Cum. masked GDD')
    ax.legend(fontsize=7.5)

    # 5. Bloom probability
    ax = axes[4]
    ax.plot(days, ps, color=MODEL_COLOR, lw=2.0)
    ax.fill_between(days, ps, color=MODEL_COLOR, alpha=0.12)
    ax.axhline(0.5, color='#555555', lw=0.8, ls=':')
    if obs_ix is not None:
        ax.axvline(obs_ix,  color='black',       lw=1.2, ls='-',  alpha=0.7,
                   label=f'Observed (day {obs_ix})')
    ax.axvline(pred_ix, color=MODEL_COLOR, lw=1.2, ls='--', alpha=0.9,
               label=f'Predicted (day {pred_ix})')
    _style_ax(ax, n, ylabel='ps(t)', xlabel=True)
    ax.set_ylim(-0.05, 1.1)
    ax.legend(fontsize=7.5, loc='upper left')

    plt.tight_layout()
    plt.show()


for year, lbl in zip(years_to_show, labels_to_show):
    sample   = pick_sample(ds_tst, year)
    obs_date = sample['observations'][OBS_KEY]
    obs_ix   = int((np.datetime64(obs_date, 'D')
                    - np.datetime64(sample['season_start'], 'D'))
                   / np.timedelta64(1, 'D'))
    plot_season_dynamics(
        model, sample, obs_ix=obs_ix,
        title=f'GlobalBetaGDDModel — {lbl}  (year {year})',
    )

## 7. Sensitivity analysis — synthetic winter temperature sweep

Generate synthetic seasonal temperature series across a range of mean winter
temperatures and record the predicted bloom day.  This reveals the model's
sensitivity curve: how much earlier does bloom shift per °C of warming?

In [ ]:
def generate_temperature_series(
    n_days: int = 365,
    mean_winter_temp: float = 4.0,
    mean_summer_temp: float = 20.0,
    noise_std: float = 2.0,
    seed: int = 0,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    days = np.arange(n_days)
    mid_winter_day = 92   # ~Jan 1 from Oct 1
    phase     = 2 * np.pi * (days - mid_winter_day) / 365.0
    amplitude = (mean_summer_temp - mean_winter_temp) / 2.0
    baseline  = (mean_summer_temp + mean_winter_temp) / 2.0
    seasonal  = baseline - amplitude * np.cos(phase)
    return (seasonal + rng.normal(0, noise_std, size=n_days)).astype(np.float32)


def make_synthetic_sample(temps: np.ndarray, year: int = 2000) -> dict:
    return {
        KEY_FEATURES: {'temperature_2m_mean': temps},
        'season_start': np.datetime64(f'{year}-10-01', 'D'),
        'year': year,
    }


@torch.no_grad()
def predict_bloom_day(model, sample) -> int:
    model.eval()
    device = next(model.parameters()).device
    t_s   = GlobalBetaGDDModel.cast_to_tensor(sample, device=device)
    batch = GlobalBetaGDDModel.collate_fn([t_s])
    ixs, _ = model(batch)
    return int(ixs[0].item() + 0.5)


WINTER_TEMPS = np.arange(0.0, 17.0, 0.5)
N_SEEDS = 10

bloom_days = []
for mw in WINTER_TEMPS:
    seeds = [predict_bloom_day(model, make_synthetic_sample(
        generate_temperature_series(mean_winter_temp=mw, seed=s)))
        for s in range(N_SEEDS)]
    bloom_days.append(np.median(seeds))
bloom_days = np.array(bloom_days)
print('Sweep done.')

In [ ]:
_MONTH_REFS = [(92, 'Jan'), (153, 'Mar'), (181, 'Apr'), (212, 'May')]

fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle('Predicted bloom day vs mean winter temperature\n'
             '(median over 10 synthetic seasons)',
             fontsize=12, fontweight='bold')

for doy, month in _MONTH_REFS:
    ax.axhline(doy, color='lightgrey', lw=0.7, zorder=0)
    ax.text(16.4, doy, month, fontsize=7.5, va='center', color='grey')

valid = bloom_days < 364
ax.plot(WINTER_TEMPS[valid], bloom_days[valid],
        color=MODEL_COLOR, lw=2.5, label='GlobalBetaGDDModel')
ax.scatter(WINTER_TEMPS[valid], bloom_days[valid],
           color=MODEL_COLOR, s=18, zorder=3)

ax.set_xlabel('Mean winter temperature (°C)', fontsize=10)
ax.set_ylabel('Predicted bloom day (days from Oct 1)', fontsize=10)
ax.legend(fontsize=9, framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, 16.5)
plt.tight_layout()
plt.show()

# Sensitivity: days earlier per °C of warming in the 4–12°C range
mask = (WINTER_TEMPS >= 4) & (WINTER_TEMPS <= 12) & valid
if mask.sum() > 1:
    coef = np.polyfit(WINTER_TEMPS[mask], bloom_days[mask], 1)
    print(f'Sensitivity (4–12 °C range): {coef[0]:.1f} days / °C  '
          f'({-coef[0]:.1f} days earlier per 1 °C warming)')

## 8. Probability curves across synthetic winter temperatures

Show how the bloom probability curve `ps(t)` shifts across the winter
temperature sweep — the beta-GDD analogue of the envelope plots in the LSTM
notebook.

In [ ]:
SWEEP_TEMPS = np.arange(0.0, 17.0, 2.0)
cmap = plt.cm.coolwarm_r
norm = plt.Normalize(SWEEP_TEMPS.min(), SWEEP_TEMPS.max())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Beta-GDD probability curves across winter temperatures  (seed 42)',
             fontsize=12, fontweight='bold')
ax_ps, ax_diff = axes

model.eval()
with torch.no_grad():
    for mw in SWEEP_TEMPS:
        ts     = generate_temperature_series(mean_winter_temp=mw, seed=42)
        sample = make_synthetic_sample(ts)
        device = next(model.parameters()).device
        t_s    = GlobalBetaGDDModel.cast_to_tensor(sample, device=device)
        batch  = GlobalBetaGDDModel.collate_fn([t_s])
        ixs, info = model(batch)
        ps     = info['ps'][0].cpu().numpy()
        pred   = int(ixs[0].item() + 0.5)
        n      = len(ts)
        days   = np.arange(n)
        diff   = np.diff(ps, prepend=ps[0])
        color  = cmap(norm(mw))

        ax_ps.plot(days, ps, color=color, lw=1.5, alpha=0.85,
                   label=f'{mw:.0f} °C  (day {pred})')
        ax_diff.plot(days, diff, color=color, lw=1.5, alpha=0.85)

ax_ps.axhline(0.5, color='#555555', lw=0.8, ls=':')
_style_ax(ax_ps, n, ylabel='P(bloom by day t)', xlabel=True)
ax_ps.set_ylim(-0.05, 1.1)
ax_ps.set_title('Cumulative bloom probability', fontsize=10)
ax_ps.legend(fontsize=7.5, ncol=2, framealpha=0.85, title='Winter temp')

ax_diff.axhline(0, color='#cccccc', lw=0.8)
_style_ax(ax_diff, n, ylabel='dP/dt', xlabel=True)
ax_diff.set_title('First difference (event probability density)', fontsize=10)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax_diff, label='Mean winter temp (°C)', shrink=0.85)

plt.tight_layout()
plt.show()